# Assignment 1 — VIIRS Active Fires in Peru (2012–2024)

**Grupo 3**  
**Integrantes:** _pendiente de completar_

Notebook reproducible para las cinco tareas del Assignment 1. La **Fase 1**
ejecuta únicamente las validaciones piloto con 2024 y las Tareas 1–2.

## 1. Consigna oficial

Fuente normativa: [GitHub issue #9 — Assignment 1](https://github.com/anzonyquispe/GeoAgent/issues/9).

La unidad observada es una **detección VIIRS**, no un incendio único. El VCF es
una línea base de cobertura arbórea correspondiente a 2003.

## 2. Entorno y versiones

Este notebook no instala paquetes. Debe ejecutarse exclusivamente en el entorno
Conda `geoagent312` mediante `conda run -n geoagent312 ...`.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import folium
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from IPython.display import display

NOTEBOOK_START = time.perf_counter()

versions = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "geopandas": gpd.__version__,
    "rasterio": rasterio.__version__,
    "folium": folium.__version__,
    "matplotlib": matplotlib.__version__,
    "requests": requests.__version__,
}
assert sys.version_info[:2] == (3, 12), (
    f"Se requiere Python 3.12; se detectó {sys.version.split()[0]}"
)
display(pd.Series(versions, name="versión"))

python                          3.12.13
platform      Windows-11-10.0.26200-SP0
pandas                            3.0.3
geopandas                         1.1.4
rasterio                          1.5.0
folium                           0.20.0
matplotlib                       3.11.0
requests                         2.34.2
Name: versión, dtype: str

## 3. Configuración mediante rutas relativas

In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(
        "No se encontró pyproject.toml en el directorio actual ni en sus padres."
    )


REPO_ROOT = find_repo_root()
ASSIGNMENT_DIR = REPO_ROOT / "Assignments" / "Assignment1"
CACHE_DIR = ASSIGNMENT_DIR / "cache_grupo3"
RAW_VIIRS_DIR = CACHE_DIR / "raw" / "viirs"
RAW_VCF_DIR = CACHE_DIR / "raw" / "vcf"
RAW_BOUNDARIES_DIR = CACHE_DIR / "raw" / "boundaries"
INTERIM_DIR = CACHE_DIR / "interim"
MANIFEST_DIR = CACHE_DIR / "manifests"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs_grupo3"

for directory in (
    RAW_VIIRS_DIR,
    RAW_VCF_DIR,
    RAW_BOUNDARIES_DIR,
    INTERIM_DIR,
    MANIFEST_DIR,
    OUTPUT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

DOWNLOAD_MANIFEST = MANIFEST_DIR / "downloads.json"
PHASE1_SUMMARY = MANIFEST_DIR / "phase1_summary.json"

print("Raíz del repositorio detectada mediante pyproject.toml:", REPO_ROOT.name)
print("CACHE_DIR:", CACHE_DIR.relative_to(REPO_ROOT))
print("OUTPUT_DIR:", OUTPUT_DIR.relative_to(REPO_ROOT))

Raíz del repositorio detectada mediante pyproject.toml: GeoAgent
CACHE_DIR: Assignments\Assignment1\cache_grupo3
OUTPUT_DIR: Assignments\Assignment1\outputs_grupo3


## 4. Catálogo de fuentes

In [3]:
ISSUE_URL = "https://github.com/anzonyquispe/GeoAgent/issues/9"
VIIRS_2024_URL = (
    "https://firms.modaps.eosdis.nasa.gov/data/country/viirs-snpp/"
    "2024/viirs-snpp_2024_Peru.csv"
)
GADM_L1_URL = (
    "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_PER_1.json"
)
GADM_L3_URL = (
    "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_PER_3.json"
)
VCF_URL = (
    "https://www.dropbox.com/scl/fi/a3c2tdgb7urvbxrmqjaox/"
    "MOD44B.061_Percent_Tree_Cover_2003Peru.tif"
    "?rlkey=1v8v271e4gn2mnnwhrc3gbp4f&st=qs07al4c&dl=0"
)

PILOT_SOURCES = {
    "viirs_2024": {
        "url": VIIRS_2024_URL,
        "path": RAW_VIIRS_DIR / "viirs-snpp_2024_Peru.csv",
        "kind": "csv",
    },
    "gadm_l1": {
        "url": GADM_L1_URL,
        "path": RAW_BOUNDARIES_DIR / "gadm41_PER_1.json",
        "kind": "json",
    },
    "gadm_l3": {
        "url": GADM_L3_URL,
        "path": RAW_BOUNDARIES_DIR / "gadm41_PER_3.json",
        "kind": "json",
    },
    "vcf_2003": {
        "url": VCF_URL,
        "path": RAW_VCF_DIR / "MOD44B.061_Percent_Tree_Cover_2003Peru.tif",
        "kind": "tif",
    },
}

pd.DataFrame(
    [
        {
            "dataset": name,
            "url": item["url"],
            "ruta_relativa": item["path"].relative_to(REPO_ROOT).as_posix(),
            "tipo": item["kind"],
        }
        for name, item in PILOT_SOURCES.items()
    ]
)

,dataset,url,ruta_relativa,tipo
0,viirs_2024,https://firms.modaps.eosdis.nasa.gov/data/coun...,Assignments/Assignment1/cache_grupo3/raw/viirs...,csv
1,gadm_l1,https://geodata.ucdavis.edu/gadm/gadm4.1/json/...,Assignments/Assignment1/cache_grupo3/raw/bound...,json
2,gadm_l3,https://geodata.ucdavis.edu/gadm/gadm4.1/json/...,Assignments/Assignment1/cache_grupo3/raw/bound...,json
3,vcf_2003,https://www.dropbox.com/scl/fi/a3c2tdgb7urvbxr...,Assignments/Assignment1/cache_grupo3/raw/vcf/M...,tif


## 5. Funciones auxiliares de adquisición y validación

In [4]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _load_manifest(path: Path) -> dict:
    if not path.exists():
        return {"files": {}}
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as exc:
        raise RuntimeError(f"Manifiesto inválido: {path}: {exc}") from exc
    if not isinstance(data, dict) or not isinstance(data.get("files"), dict):
        raise RuntimeError(f"Estructura de manifiesto inválida: {path}")
    return data


def _write_manifest(path: Path, manifest: dict) -> None:
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temp.replace(path)


def validate_file_signature(path: Path, expected_kind: str) -> None:
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Archivo ausente o vacío: {path}")

    with path.open("rb") as stream:
        head = stream.read(8192)
    stripped = head.lstrip()
    lowered = stripped[:512].lower()
    html_markers = (b"<!doctype html", b"<html", b"<head", b"<body")
    if any(lowered.startswith(marker) for marker in html_markers):
        raise RuntimeError(
            f"La descarga parece HTML, no {expected_kind.upper()}: {path}"
        )

    if expected_kind == "csv":
        first_line = head.decode("utf-8-sig", errors="replace").splitlines()[0]
        required = {"latitude", "longitude", "acq_date", "frp"}
        columns = {item.strip() for item in first_line.split(",")}
        if not required.issubset(columns):
            raise RuntimeError(
                f"Cabecera CSV inesperada en {path}; faltan {sorted(required - columns)}"
            )
    elif expected_kind == "json":
        if not stripped.startswith((b"{", b"[")):
            raise RuntimeError(f"Firma JSON inválida: {path}")
    elif expected_kind == "tif":
        classic_tiff = (b"II*\x00", b"MM\x00*")
        big_tiff = (b"II+\x00", b"MM\x00+")
        if not head.startswith(classic_tiff + big_tiff):
            raise RuntimeError(f"Firma TIFF/BigTIFF inválida: {path}")
    else:
        raise ValueError(f"Tipo de archivo no soportado: {expected_kind}")


def download_stream(
    url: str,
    destination: Path,
    expected_kind: str,
    manifest_path: Path = DOWNLOAD_MANIFEST,
    chunk_size: int = 1024 * 1024,
) -> dict:
    destination.parent.mkdir(parents=True, exist_ok=True)
    key = destination.relative_to(REPO_ROOT).as_posix()
    manifest = _load_manifest(manifest_path)
    previous = manifest["files"].get(key)

    if destination.exists():
        validate_file_signature(destination, expected_kind)
        size = destination.stat().st_size
        checksum = sha256_file(destination)
        if (
            previous
            and previous.get("requested_url") == url
            and previous.get("size_bytes") == size
            and previous.get("sha256") == checksum
        ):
            record = dict(previous)
            record["reused"] = True
            record["checked_at_utc"] = datetime.now(timezone.utc).isoformat()
            return record

        record = {
            "requested_url": url,
            "resolved_url": previous.get("resolved_url") if previous else url,
            "downloaded_at_utc": (
                previous.get("downloaded_at_utc")
                if previous
                else datetime.now(timezone.utc).isoformat()
            ),
            "checked_at_utc": datetime.now(timezone.utc).isoformat(),
            "size_bytes": size,
            "sha256": checksum,
            "content_type": previous.get("content_type") if previous else None,
            "reused": True,
        }
        manifest["files"][key] = record
        _write_manifest(manifest_path, manifest)
        return record

    partial = destination.with_suffix(destination.suffix + ".part")
    partial.unlink(missing_ok=True)
    started = time.perf_counter()
    try:
        session = requests.Session()
        response = session.get(
            url,
            stream=True,
            allow_redirects=True,
            timeout=(20, 180),
            headers={"User-Agent": "GeoAgent-Assignment1-Grupo3/1.0"},
        )
        initial_content_type = response.headers.get("Content-Type", "")
        if expected_kind == "tif" and "text/html" in initial_content_type.lower():
            response.close()
            redirect_probe = session.head(
                url,
                allow_redirects=False,
                timeout=(20, 60),
                headers={"User-Agent": "curl/8.14.1"},
            )
            redirect_url = redirect_probe.headers.get("Location")
            redirect_probe.close()
            if not redirect_url:
                raise RuntimeError(
                    "El enlace VCF oficial devolvió HTML y no publicó una "
                    "redirección de contenido en la cabecera HEAD."
                )
            response = session.get(
                redirect_url,
                stream=True,
                allow_redirects=True,
                timeout=(20, 180),
                headers={"User-Agent": "GeoAgent-Assignment1-Grupo3/1.0"},
            )

        with response:
            response.raise_for_status()
            content_type = response.headers.get("Content-Type", "")
            iterator = response.iter_content(chunk_size=chunk_size)
            first_chunk = next(iterator, b"")
            if not first_chunk:
                raise RuntimeError(f"Respuesta vacía al descargar {url}")
            lowered = first_chunk.lstrip()[:512].lower()
            if "text/html" in content_type.lower() or lowered.startswith(
                (b"<!doctype html", b"<html")
            ):
                raise RuntimeError(
                    f"El servidor devolvió HTML ({content_type}) para {url}"
                )
            with partial.open("wb") as stream:
                stream.write(first_chunk)
                for chunk in iterator:
                    if chunk:
                        stream.write(chunk)

            validate_file_signature(partial, expected_kind)
            partial.replace(destination)
            record = {
                "requested_url": url,
                "resolved_url": str(response.url),
                "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
                "checked_at_utc": datetime.now(timezone.utc).isoformat(),
                "size_bytes": destination.stat().st_size,
                "sha256": sha256_file(destination),
                "content_type": content_type,
                "elapsed_seconds": round(time.perf_counter() - started, 3),
                "reused": False,
            }
    except Exception:
        partial.unlink(missing_ok=True)
        raise

    manifest["files"][key] = record
    _write_manifest(manifest_path, manifest)
    return record

## 6. Adquisición piloto

In [5]:
download_records = {}
for dataset in ("viirs_2024", "gadm_l1", "gadm_l3"):
    source = PILOT_SOURCES[dataset]
    download_records[dataset] = download_stream(
        source["url"], source["path"], source["kind"]
    )

try:
    source = PILOT_SOURCES["vcf_2003"]
    download_records["vcf_2003"] = download_stream(
        source["url"], source["path"], source["kind"]
    )
except Exception as exc:
    manual_instruction = (
        "Descarga manual requerida: abre exclusivamente el enlace VCF del issue #9, "
        "descarga el archivo sin convertirlo ni renombrarlo y guárdalo en "
        f"'{PILOT_SOURCES['vcf_2003']['path'].relative_to(REPO_ROOT).as_posix()}'. "
        "Luego vuelve a ejecutar esta celda. No uses otra fuente."
    )
    raise RuntimeError(f"No fue posible automatizar el VCF: {exc}. {manual_instruction}") from exc

download_table = pd.DataFrame.from_dict(download_records, orient="index")
display(download_table[["size_bytes", "sha256", "content_type", "reused"]])

,size_bytes,sha256,content_type,reused
viirs_2024,7963113,21afe44401c95f7988950c425d23938dbaffd98d392b87...,text/csv,True
gadm_l1,695822,1ae8d736d0307a39169a300539cdac07a2a7d6d91256e3...,application/json,True
gadm_l3,4425168,d684f8b2c8809305672727856d74e399eddf50f63fcdf2...,application/json,True
vcf_2003,32784639,cb5c56e45f332b269c66d0d897f7c5908005d65294aff4...,image/tiff,True


## 7. Validación territorial

In [6]:
departments = gpd.read_file(PILOT_SOURCES["gadm_l1"]["path"])
districts = gpd.read_file(PILOT_SOURCES["gadm_l3"]["path"])

required_l1 = {"GID_1", "NAME_1", "geometry"}
required_l3 = {"GID_3", "GID_1", "NAME_3", "NAME_1", "geometry"}
missing_l1 = required_l1 - set(departments.columns)
missing_l3 = required_l3 - set(districts.columns)
if missing_l1 or missing_l3:
    raise ValueError(
        f"Contrato territorial incumplido: L1 faltan {sorted(missing_l1)}; "
        f"L3 faltan {sorted(missing_l3)}"
    )
if departments.crs is None or districts.crs is None:
    raise ValueError("GADM debe declarar CRS en ambos niveles.")

territorial_qc = {
    "department_rows": int(len(departments)),
    "district_rows": int(len(districts)),
    "department_crs": str(departments.crs),
    "district_crs": str(districts.crs),
    "department_empty": int(departments.geometry.is_empty.sum()),
    "district_empty": int(districts.geometry.is_empty.sum()),
    "department_missing_geometry": int(departments.geometry.isna().sum()),
    "district_missing_geometry": int(districts.geometry.isna().sum()),
    "department_invalid_before": int((~departments.geometry.is_valid).sum()),
    "district_invalid_before": int((~districts.geometry.is_valid).sum()),
    "duplicate_gid1": int(departments["GID_1"].duplicated().sum()),
    "duplicate_gid3": int(districts["GID_3"].duplicated().sum()),
    "district_missing_parent_gid1": int(districts["GID_1"].isna().sum()),
}

if territorial_qc["department_invalid_before"]:
    departments = departments.copy()
    departments["geometry"] = departments.geometry.make_valid()
if territorial_qc["district_invalid_before"]:
    districts = districts.copy()
    districts["geometry"] = districts.geometry.make_valid()

territorial_qc["department_invalid_after"] = int(
    (~departments.geometry.is_valid).sum()
)
territorial_qc["district_invalid_after"] = int(
    (~districts.geometry.is_valid).sum()
)

parent_ids = set(districts["GID_1"].dropna().astype(str))
department_ids = set(departments["GID_1"].dropna().astype(str))
territorial_qc["district_parents_missing_in_l1"] = sorted(
    parent_ids - department_ids
)
territorial_qc["l1_without_l3_districts"] = sorted(department_ids - parent_ids)
territorial_qc["gid3_parent_conflicts"] = int(
    (districts.groupby("GID_3", dropna=False)["GID_1"].nunique() > 1).sum()
)

departments_equal_area = departments.to_crs("EPSG:6933")
districts_equal_area = districts.to_crs("EPSG:6933")
department_union = departments_equal_area.geometry.union_all()
district_union = districts_equal_area.geometry.union_all()
territorial_qc["district_coverage_of_departments"] = float(
    district_union.intersection(department_union).area / department_union.area
)
territorial_qc["district_area_outside_departments"] = float(
    district_union.difference(department_union).area / district_union.area
)
territorial_qc["department_bounds"] = departments.total_bounds.tolist()
territorial_qc["district_bounds"] = districts.total_bounds.tolist()

assert territorial_qc["department_empty"] == 0
assert territorial_qc["district_empty"] == 0
assert territorial_qc["department_missing_geometry"] == 0
assert territorial_qc["district_missing_geometry"] == 0
assert territorial_qc["duplicate_gid1"] == 0
assert territorial_qc["duplicate_gid3"] == 0
assert territorial_qc["district_missing_parent_gid1"] == 0
assert territorial_qc["gid3_parent_conflicts"] == 0
assert not territorial_qc["district_parents_missing_in_l1"]
assert territorial_qc["department_invalid_after"] == 0
assert territorial_qc["district_invalid_after"] == 0
if territorial_qc["district_coverage_of_departments"] < 0.99:
    warnings.warn("La cobertura L3 sobre L1 es inferior a 99%.")

print("Columnas GADM L1:", departments.columns.tolist())
print("Columnas GADM L3:", districts.columns.tolist())
display(pd.Series(territorial_qc, name="valor"))

Columnas GADM L1: ['GID_1', 'GID_0', 'COUNTRY', 'NAME_1', 'VARNAME_1', 'NL_NAME_1', 'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'geometry']
Columnas GADM L3: ['GID_3', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'GID_2', 'NAME_2', 'NL_NAME_2', 'NAME_3', 'VARNAME_3', 'NL_NAME_3', 'TYPE_3', 'ENGTYPE_3', 'CC_3', 'HASC_3', 'geometry']


department_rows                                                          26
district_rows                                                          1815
department_crs                                                    EPSG:4326
district_crs                                                      EPSG:4326
department_empty                                                          0
district_empty                                                            0
department_missing_geometry                                               0
district_missing_geometry                                                 0
department_invalid_before                                                 0
district_invalid_before                                                   0
duplicate_gid1                                                            0
duplicate_gid3                                                            0
district_missing_parent_gid1                                              0
department_i

## 8. Validación VIIRS 2024

In [7]:
required_viirs = {
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument",
    "confidence",
    "frp",
}
fires_2024 = pd.read_csv(
    PILOT_SOURCES["viirs_2024"]["path"],
    dtype={
        "acq_time": "string",
        "satellite": "string",
        "instrument": "string",
        "confidence": "string",
    },
)
missing_viirs = required_viirs - set(fires_2024.columns)
if missing_viirs:
    raise ValueError(f"Faltan columnas VIIRS obligatorias: {sorted(missing_viirs)}")

for numeric_column in ("latitude", "longitude", "frp"):
    fires_2024[numeric_column] = pd.to_numeric(
        fires_2024[numeric_column], errors="coerce"
    )
parsed_dates = pd.to_datetime(fires_2024["acq_date"], errors="coerce")

time_raw = fires_2024["acq_time"].astype("string").str.strip()
time_digits = time_raw.str.fullmatch(r"\d{1,4}", na=False)
time_hhmm = time_raw.str.zfill(4)
hours = pd.to_numeric(time_hhmm.str[:2], errors="coerce")
minutes = pd.to_numeric(time_hhmm.str[2:], errors="coerce")
valid_hhmm = time_digits & hours.between(0, 23) & minutes.between(0, 59)
fires_2024["acq_time_hhmm"] = time_hhmm

confidence_raw = fires_2024["confidence"].astype("string").str.strip().str.lower()
confidence_map = {
    "h": "high",
    "high": "high",
    "n": "nominal",
    "nominal": "nominal",
    "l": "low",
    "low": "low",
}
fires_2024["confidence_raw"] = confidence_raw
fires_2024["confidence_norm"] = confidence_raw.map(confidence_map).astype("string")
unknown_confidence = sorted(
    set(confidence_raw.dropna().unique()) - set(confidence_map)
)

viirs_qc = {
    "rows": int(len(fires_2024)),
    "columns": fires_2024.columns.tolist(),
    "dtypes": {column: str(dtype) for column, dtype in fires_2024.dtypes.items()},
    "date_min": parsed_dates.min().date().isoformat(),
    "date_max": parsed_dates.max().date().isoformat(),
    "invalid_dates": int(parsed_dates.isna().sum()),
    "rows_outside_2024": int((parsed_dates.dt.year != 2024).sum()),
    "invalid_latitude": int(
        (fires_2024["latitude"].isna() | ~fires_2024["latitude"].between(-90, 90)).sum()
    ),
    "invalid_longitude": int(
        (fires_2024["longitude"].isna() | ~fires_2024["longitude"].between(-180, 180)).sum()
    ),
    "invalid_frp": int(
        (fires_2024["frp"].isna() | ~np.isfinite(fires_2024["frp"])).sum()
    ),
    "nonpositive_frp": int((fires_2024["frp"] <= 0).sum()),
    "invalid_acq_time": int((~valid_hhmm).sum()),
    "confidence_counts": confidence_raw.value_counts(dropna=False).to_dict(),
    "unknown_confidence": unknown_confidence,
    "satellite_counts": fires_2024["satellite"].value_counts(dropna=False).to_dict(),
    "instrument_counts": fires_2024["instrument"].value_counts(dropna=False).to_dict(),
    "duplicate_rows": int(fires_2024.duplicated().sum()),
    "null_counts": {
        column: int(count)
        for column, count in fires_2024.isna().sum().items()
        if count
    },
    "latitude_min": float(fires_2024["latitude"].min()),
    "latitude_max": float(fires_2024["latitude"].max()),
    "longitude_min": float(fires_2024["longitude"].min()),
    "longitude_max": float(fires_2024["longitude"].max()),
    "frp_min": float(fires_2024["frp"].min()),
    "frp_max": float(fires_2024["frp"].max()),
}

assert viirs_qc["invalid_dates"] == 0
assert viirs_qc["rows_outside_2024"] == 0
assert viirs_qc["invalid_latitude"] == 0
assert viirs_qc["invalid_longitude"] == 0
assert viirs_qc["invalid_frp"] == 0
assert viirs_qc["invalid_acq_time"] == 0
assert not unknown_confidence
assert fires_2024["confidence_norm"].notna().all()

print("Esquema VIIRS 2024 original:", [c for c in fires_2024.columns if c not in {"acq_time_hhmm", "confidence_raw", "confidence_norm"}])
display(pd.Series(viirs_qc, name="valor"))

Esquema VIIRS 2024 original: ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type']


rows                                                              99932
columns               [latitude, longitude, bright_ti4, scan, track,...
dtypes                {'latitude': 'float64', 'longitude': 'float64'...
date_min                                                     2024-01-01
date_max                                                     2024-12-31
invalid_dates                                                         0
rows_outside_2024                                                     0
invalid_latitude                                                      0
invalid_longitude                                                     0
invalid_frp                                                           0
nonpositive_frp                                                       0
invalid_acq_time                                                      0
confidence_counts                    {'n': 88096, 'l': 6723, 'h': 5113}
unknown_confidence                                              

## 9. Validación VCF 2003

In [8]:
vcf_path = PILOT_SOURCES["vcf_2003"]["path"]
valid_pixels = 0
masked_pixels = 0
vcf_min = None
vcf_max = None

with rasterio.open(vcf_path) as src:
    if src.count != 1:
        raise ValueError(f"El VCF debe tener una banda; se detectaron {src.count}.")
    if src.crs is None:
        raise ValueError("El VCF no declara CRS.")

    for _, window in src.block_windows(1):
        block = src.read(1, window=window, masked=True)
        valid = block.compressed()
        valid_pixels += int(valid.size)
        masked_pixels += int(block.size - valid.size)
        if valid.size:
            block_min = float(valid.min())
            block_max = float(valid.max())
            vcf_min = block_min if vcf_min is None else min(vcf_min, block_min)
            vcf_max = block_max if vcf_max is None else max(vcf_max, block_max)

    dtype = np.dtype(src.dtypes[0])
    band_uncompressed_bytes = int(src.width * src.height * dtype.itemsize)
    mask_estimated_bytes = int(src.width * src.height)
    vcf_qc = {
        "driver": src.driver,
        "band_count": int(src.count),
        "crs": str(src.crs),
        "width": int(src.width),
        "height": int(src.height),
        "resolution": [float(abs(src.res[0])), float(abs(src.res[1]))],
        "dtype": str(dtype),
        "nodata": None if src.nodata is None else float(src.nodata),
        "mask_flags": [str(flag) for flag in src.mask_flag_enums[0]],
        "block_shape": list(src.block_shapes[0]),
        "compression": src.profile.get("compress"),
        "valid_pixels": valid_pixels,
        "masked_pixels": masked_pixels,
        "valid_min": vcf_min,
        "valid_max": vcf_max,
        "band_uncompressed_bytes": band_uncompressed_bytes,
        "mask_estimated_bytes": mask_estimated_bytes,
        "file_size_bytes": vcf_path.stat().st_size,
    }

if valid_pixels == 0:
    raise ValueError("El VCF no contiene píxeles válidos.")
display(pd.Series(vcf_qc, name="valor"))

driver                                                                 GTiff
band_count                                                                 1
crs                        PROJCS["unknown",GEOGCS["unknown",DATUM["unkno...
width                                                                   8480
height                                                                  9012
resolution                                  [231.6563582625, 231.6563582625]
dtype                                                                  uint8
nodata                                                                 253.0
mask_flags                                                               [8]
block_shape                                                       [512, 512]
compression                                                              lzw
valid_pixels                                                        59235695
masked_pixels                                                       17186065

## 10. Tarea 1 — Coropleta estática por distrito

Se realiza point-in-polygon en CRS consistente, se conservan todos los
distritos y se completan con cero aquellos sin detecciones.

In [9]:
valid_points = fires_2024[
    fires_2024["latitude"].between(-90, 90)
    & fires_2024["longitude"].between(-180, 180)
].copy()
fires_points = gpd.GeoDataFrame(
    valid_points,
    geometry=gpd.points_from_xy(valid_points["longitude"], valid_points["latitude"]),
    crs="EPSG:4326",
)
districts_join = districts.to_crs(fires_points.crs)
join_columns = ["GID_3", "GID_1", "NAME_3", "NAME_1", "geometry"]
assigned = gpd.sjoin(
    fires_points,
    districts_join[join_columns],
    how="left",
    predicate="within",
)
raw_matches = assigned.dropna(subset=["GID_3"]).copy()
matches_per_point = raw_matches.groupby(level=0).size()
conflict_indices = matches_per_point[matches_per_point > 1].index
overlap_conflict_points = int(len(conflict_indices))
overlap_extra_matches = int((matches_per_point - 1).clip(lower=0).sum())

# GADM L3 contiene pequeños solapamientos. Como dos polígonos del mismo
# nivel no ofrecen una prioridad semántica, los puntos ambiguos se excluyen
# en vez de asignarlos arbitrariamente o contarlos dos veces.
matched = raw_matches.loc[~raw_matches.index.isin(conflict_indices)].copy()
unmatched_no_polygon = int(
    len(fires_points) - raw_matches.index.nunique()
)
district_counts = matched.groupby("GID_3").size().rename("detecciones_2024")
district_map = districts.merge(district_counts, on="GID_3", how="left")
district_map["detecciones_2024"] = (
    district_map["detecciones_2024"].fillna(0).astype("int64")
)

map_sum = int(district_map["detecciones_2024"].sum())
assigned_count = int(len(matched))
if map_sum != assigned_count:
    raise AssertionError(
        f"Conservación incumplida: mapa={map_sum}, asignados={assigned_count}"
    )

tarea1_output = OUTPUT_DIR / "tarea1_mapa_incendios_2024.png"
fig, ax = plt.subplots(figsize=(11, 14))
district_map.plot(
    column="detecciones_2024",
    cmap="OrRd",
    linewidth=0.12,
    edgecolor="#8a8a8a",
    legend=True,
    legend_kwds={"label": "Detecciones VIIRS 2024", "shrink": 0.65},
    ax=ax,
)
departments.to_crs(district_map.crs).boundary.plot(
    ax=ax, color="#303030", linewidth=0.55
)
ax.set_title(
    "Perú: detecciones de incendios VIIRS por distrito, 2024",
    fontsize=15,
    pad=14,
)
ax.set_axis_off()
fig.text(
    0.5,
    0.04,
    "Fuente: NASA FIRMS VIIRS-SNPP; límites provisionales GADM 4.1",
    ha="center",
    fontsize=9,
)
fig.savefig(tarea1_output, dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig)

task1_qc = {
    "input_points": int(len(fires_points)),
    "assigned_points": assigned_count,
    "unassigned_no_polygon": unmatched_no_polygon,
    "overlap_conflict_points_excluded": overlap_conflict_points,
    "overlap_extra_matches": overlap_extra_matches,
    "unassigned_total": unmatched_no_polygon + overlap_conflict_points,
    "districts_total": int(len(district_map)),
    "districts_with_zero": int((district_map["detecciones_2024"] == 0).sum()),
    "map_sum": map_sum,
    "output": tarea1_output.relative_to(REPO_ROOT).as_posix(),
}
display(pd.Series(task1_qc, name="valor"))

input_points                                                                    99932
assigned_points                                                                 99717
unassigned_no_polygon                                                             187
overlap_conflict_points_excluded                                                   28
overlap_extra_matches                                                              28
unassigned_total                                                                  215
districts_total                                                                  1815
districts_with_zero                                                               316
map_sum                                                                         99717
output                              Assignments/Assignment1/outputs_grupo3/tarea1_...
Name: valor, dtype: object

## 11. Tarea 2 — Top 100 FRP en mapa dinámico

Se usa `folium.Circle`. Para conservar proporcionalidad estricta se define una
única constante para todos los puntos:

`radio_m = FRP_RADIUS_M_PER_MW × frp`

La constante se calibra para que el mayor FRP tenga un radio de 25 km. No se
suma un radio mínimo.

In [10]:
fire_candidates = fires_2024[
    fires_2024["latitude"].between(-90, 90)
    & fires_2024["longitude"].between(-180, 180)
    & fires_2024["frp"].notna()
    & np.isfinite(fires_2024["frp"])
    & (fires_2024["frp"] > 0)
].copy()
if len(fire_candidates) < 100:
    raise ValueError(f"Solo hay {len(fire_candidates)} candidatos válidos; se requieren 100.")

top100 = fire_candidates.sort_values(
    ["frp", "acq_date", "acq_time_hhmm"],
    ascending=[False, True, True],
    kind="mergesort",
).head(100).copy()

TARGET_MAX_RADIUS_M = 25_000.0
FRP_RADIUS_M_PER_MW = TARGET_MAX_RADIUS_M / float(top100["frp"].max())
top100["radius_m"] = top100["frp"] * FRP_RADIUS_M_PER_MW

if len(top100) != 100:
    raise AssertionError("La selección Top 100 no contiene exactamente 100 filas.")
if not np.allclose(
    top100["radius_m"] / top100["frp"], FRP_RADIUS_M_PER_MW
):
    raise AssertionError("El radio no es estrictamente proporcional a FRP.")

fire_map = folium.Map(
    location=[float(top100["latitude"].mean()), float(top100["longitude"].mean())],
    zoom_start=5,
    tiles="CartoDB positron",
    control_scale=True,
)
for row in top100.itertuples(index=False):
    popup = folium.Popup(
        (
            f"<b>Fecha:</b> {row.acq_date}<br>"
            f"<b>Hora HHMM:</b> {row.acq_time_hhmm}<br>"
            f"<b>FRP:</b> {row.frp:.2f} MW<br>"
            f"<b>Confidence:</b> {row.confidence_raw} ({row.confidence_norm})"
        ),
        max_width=280,
    )
    folium.Circle(
        location=[row.latitude, row.longitude],
        radius=float(row.radius_m),
        color="#b91c1c",
        weight=1,
        fill=True,
        fill_color="#ef4444",
        fill_opacity=0.35,
        popup=popup,
        tooltip=f"FRP {row.frp:.2f} MW",
    ).add_to(fire_map)

fire_map.fit_bounds(
    [
        [float(top100["latitude"].min()), float(top100["longitude"].min())],
        [float(top100["latitude"].max()), float(top100["longitude"].max())],
    ]
)
tarea2_output = OUTPUT_DIR / "tarea2_top100_incendios_2024.html"
fire_map.save(tarea2_output)

task2_qc = {
    "top_n": int(len(top100)),
    "frp_min_top100": float(top100["frp"].min()),
    "frp_max_top100": float(top100["frp"].max()),
    "radius_constant_m_per_mw": float(FRP_RADIUS_M_PER_MW),
    "radius_min_m": float(top100["radius_m"].min()),
    "radius_max_m": float(top100["radius_m"].max()),
    "strictly_proportional": True,
    "output": tarea2_output.relative_to(REPO_ROOT).as_posix(),
}
display(pd.Series(task2_qc, name="valor"))

top_n                                                                     100
frp_min_top100                                                         204.06
frp_max_top100                                                          645.8
radius_constant_m_per_mw                                            38.711675
radius_min_m                                                      7899.504491
radius_max_m                                                          25000.0
strictly_proportional                                                    True
output                      Assignments/Assignment1/outputs_grupo3/tarea2_...
Name: valor, dtype: object

## 12. Tarea 3 — Media VCF por distrito

**Pendiente para Fase 2; no ejecutada aquí.** Se calculará la media zonal del
VCF 2003 respetando CRS, máscara y nodata. La mediana se obtendrá solo entre
distritos con píxeles válidos y la selección será estrictamente mayor.

## 13. Tarea 4 — VCF del píxel > 50

**Pendiente para Fase 2; no ejecutada aquí.** La primera implementación será
reproducible por año y por chunks mediante `rasterio.sample`, sin `iterrows`
sobre las detecciones. Los resultados parciales se escribirán como Parquet
particionado por año. Dask quedará documentado solo como alternativa.

## 14. Tarea 5 — Serie 2013–2024 por departamento

**Pendiente para Fase 2; no ejecutada aquí.** Usará únicamente los resultados
de Tarea 4 con `confidence_norm == "high"`, agregados por `GID_1` y año, con
producto cartesiano completo de departamentos y años 2013–2024.

## 15. Control de calidad de Fase 1

In [11]:
phase1_checks = {
    "python_3_12": sys.version_info[:2] == (3, 12),
    "territorial_ids_unique": (
        territorial_qc["duplicate_gid1"] == 0
        and territorial_qc["duplicate_gid3"] == 0
    ),
    "territorial_hierarchy_complete": not territorial_qc[
        "district_parents_missing_in_l1"
    ],
    "viirs_year_is_2024": viirs_qc["rows_outside_2024"] == 0,
    "viirs_coordinates_valid": (
        viirs_qc["invalid_latitude"] == 0
        and viirs_qc["invalid_longitude"] == 0
    ),
    "viirs_frp_valid": viirs_qc["invalid_frp"] == 0,
    "viirs_acq_time_valid": viirs_qc["invalid_acq_time"] == 0,
    "confidence_normalized": not viirs_qc["unknown_confidence"],
    "vcf_single_band": vcf_qc["band_count"] == 1,
    "vcf_has_valid_pixels": vcf_qc["valid_pixels"] > 0,
    "task1_sum_conserved": task1_qc["map_sum"] == task1_qc["assigned_points"],
    "task2_exactly_100": task2_qc["top_n"] == 100,
    "task2_radius_proportional": task2_qc["strictly_proportional"],
    "task1_output_exists": tarea1_output.is_file(),
    "task2_output_exists": tarea2_output.is_file(),
}
if not all(phase1_checks.values()):
    failed = [name for name, passed in phase1_checks.items() if not passed]
    raise AssertionError(f"Fallaron controles de Fase 1: {failed}")
display(pd.Series(phase1_checks, name="aprobado"))

python_3_12                       True
territorial_ids_unique            True
territorial_hierarchy_complete    True
viirs_year_is_2024                True
viirs_coordinates_valid           True
viirs_frp_valid                   True
viirs_acq_time_valid              True
confidence_normalized             True
vcf_single_band                   True
vcf_has_valid_pixels              True
task1_sum_conserved               True
task2_exactly_100                 True
task2_radius_proportional         True
task1_output_exists               True
task2_output_exists               True
Name: aprobado, dtype: bool

## 16. Limitaciones

- GADM 4.1 es provisional y sus códigos no se denominan UBIGEO.
- El VCF representa cobertura arbórea en 2003, no cobertura actual.
- Una detección FIRMS no equivale necesariamente a un evento de incendio único.
- La Fase 1 solo descarga VIIRS 2024; no usa el CSV 2023 existente en `_data`.
- Tareas 3–5 permanecen sin ejecutar hasta autorización posterior.

## 17. Metodología

El análisis usa GeoPandas para point-in-polygon y coropletas, Rasterio para
inspección segura del VCF y Folium únicamente para visualización posterior al
cálculo. Todas las fuentes se almacenan en una caché relativa con URL, fecha,
tamaño y SHA-256. No se usan DuckDB, Dask, Spark, Sedona, Earth Engine ni
Geowombat en el camino crítico de Fase 1.

## 18. Inventario de entregables y resumen de ejecución

In [12]:
artifact_paths = [tarea1_output, tarea2_output]
artifact_inventory = [
    {
        "path": path.relative_to(REPO_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]

elapsed_seconds = round(time.perf_counter() - NOTEBOOK_START, 3)
phase1_summary = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds": elapsed_seconds,
    "versions": versions,
    "downloads": download_records,
    "territorial": territorial_qc,
    "viirs_2024": viirs_qc,
    "vcf": vcf_qc,
    "task1": task1_qc,
    "task2": task2_qc,
    "checks": phase1_checks,
    "artifacts": artifact_inventory,
}
PHASE1_SUMMARY.write_text(
    json.dumps(phase1_summary, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

display(pd.DataFrame(artifact_inventory))
print(f"Fase 1 completada en {elapsed_seconds:.3f} segundos.")
print("Resumen reproducible:", PHASE1_SUMMARY.relative_to(REPO_ROOT))

,path,size_bytes,sha256
0,Assignments/Assignment1/outputs_grupo3/tarea1_...,899434,f7557f6a1c46a09e061aaa39b2f4f4c8e5cc935112dcae...
1,Assignments/Assignment1/outputs_grupo3/tarea2_...,137061,fef41ee58898e9039df9fe519f2f130662f236f938a6b7...


Fase 1 completada en 7.338 segundos.
Resumen reproducible: Assignments\Assignment1\cache_grupo3\manifests\phase1_summary.json
